In [ ]:
def section_aware_split(text: str, max_chunk_len: int = 1500) -> list:
    """
    Chunk a Markdown-style document into hierarchical sections (using #, ##, ###) 
    and return structured chunks with section path and level.
    """
    import re

    lines = text.splitlines()
    chunks = []
    current_chunk_lines = []
    current_path = []

    def flush_chunk():
        if not current_chunk_lines:
            return
        content = "\n".join(current_chunk_lines).strip()
        if content:
            chunks.append({
                "section_path": current_path.copy(),
                "level": len(current_path),
                "content": content
            })

    for line in lines:
        header_match = re.match(r"^(#{1,6})\s+(.*)", line)
        if header_match:
            # New header found
            flush_chunk()
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_path = current_path[:level - 1] + [title]
            current_chunk_lines = [line]
        else:
            current_chunk_lines.append(line)

    flush_chunk()
    return chunks

In [ ]:
import os
import nest_asyncio

from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.vector_stores import VectorStoreQueryResult
from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings
from typing import List
from dotenv import load_dotenv
import json
import os
nest_asyncio.apply()
load_dotenv(dotenv_path=".env.dev")

In [ ]:
from docx import Document

def extract_tables_as_markdown(docx_path):
    doc = Document(docx_path)
    markdown_tables = []
    for table in doc.tables:
        rows = []
        for row in table.rows:
            cells = [cell.text.strip() for cell in row.cells]
            rows.append("| " + " | ".join(cells) + " |")
        if rows:
            header = rows[0]
            separator = "| " + " | ".join(["---"] * len(table.columns)) + " |"
            markdown_table = "\n".join([header, separator] + rows[1:])
            markdown_tables.append(markdown_table)
    return markdown_tables

## Setup Cohear Embedding service

from llama_index.core import StorageContext

In [ ]:
# … (no need to call load_dotenv() here) …

# Hard-code your key and model ID:
COHEAR_KEY      = "Iyn2rmOdEgiUKfxptJDhCKRwgfeIWhZ37sxzKUAc"
COHEAR_MODEL_ID = "embed-multilingual-light-v3.0"

print("🔑 Using Cohere key:   ", COHEAR_KEY)
print("🔢 Using Cohere model: ", COHEAR_MODEL_ID)

embed_model = CohereEmbedding(
    api_key=COHEAR_KEY,
    model_name=COHEAR_MODEL_ID,
    input_type="search_document",
    embedding_type="float",
)

Settings.chunk_size = 1024

## Innitiates VectorStore database (Qdrant)

from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
import os

# Initialize Qdrant client with HTTP (not gRPC)
client = QdrantClient(
    url="http://localhost:6433",  # Using HTTP endpoint exposed by Docker
    api_key=os.getenv("QDRANT_API_KEY"),
    prefer_grpc=False,            # Disable gRPC to avoid connection issues
    timeout=60,
    check_compatibility=False     # Suppress version mismatch warning
)

# Load collection name from environment
collection_name = os.getenv("QDRANT_COLLECTION_NAME")

# Delete collection if it exists
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

# Create Qdrant vector store with hybrid search enabled
vector_store = QdrantVectorStore(
    collection_name=collection_name,
    client=client,
    enable_hybrid=True,
    batch_size=20,
    prefer_grpc=False             # Match client setting
)

## Start embedding process.... into vector database

# ✅ Hardcoded API key and model config (no .env loading)
COHERE_API_KEY = "YsB5acLd9Szu9duW5BzCWPfDY1MiDl93MHD2ArYV"
COHERE_MODEL_ID = "embed-multilingual-light-v3.0"  # <-- replace with your actual model if different

QDRANT_URL = "http://localhost:6334"
QDRANT_API_KEY = None  # Set this to your Qdrant key if needed
COLLECTION_NAME = "my_collection"

print("✅ COHERE_API_KEY loaded.")

# ✅ Initialize Cohere embed model
embed_model = CohereEmbedding(
    cohere_api_key=COHERE_API_KEY,
    model_name=COHERE_MODEL_ID,
    input_type="search_document",
    embedding_type="float",
)

# ✅ Build index from documents
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents=documents,
    embed_model=embed_model,
    storage_context=storage_context,
)

## Try to retrive relavent nodes with question.

embed_model = CohereEmbedding(
    api_key=os.getenv("COHERE_API_KEY"),
    model_name=os.getenv("COHERE_MODEL_ID"),
    input_type="search_query",
    embedding_type="float",
)

search_query_retriever = index.as_retriever()

search_query_retrieved_nodes = search_query_retriever.retrieve(
"Do all Walmart locations offer scan & go?"
)

In [ ]:
from llama_index.core.response.notebook_utils import display_source_node
for n in search_query_retrieved_nodes:
    display_source_node(n, source_length=2000)

In [9]:
from llama_index.core.response.notebook_utils import display_source_node
for n in search_query_retrieved_nodes:
    display_source_node(n, source_length=2000)

**Node ID:** e9c37a51-1636-4c34-b331-3c040015dca9<br>**Similarity:** 0.43584468960762024<br>**Text:** #กรณีจ่ายเงินประกันแล้วแต่ไม่ได้เข้าพื้นที่จริง เนื่องจากปัญหาจากทางผู้ให้เช่า[lotus's] เช่น แผนผังหน้างานร้านไม่เป็นไปตามข้อตกลง   ลูกค้าสามารถขอคืน หรือย้ายเงินประกันไปสาขาอื่น หรือ ใช้จ่ายค่าเช่าสาขาอื่น ได้หรือไม่ และใช้เอกสารหลักฐานใดในการขอคืน หรือ ย้ายไปใช้เป้นเงินประกันสาขาอื่น หรือ จ่ายค่าเช่าสาขาอื่นภายใต้่ชื่อลูกค้าเดียวกันมีอยู่กับLotus's ได้ / เอกกสาร 

1. Memo ชี้แจงสาเหตุและอนุมัติใช้เงินประกันที่ได้รับอนุมัติจากWL3 Leasing 

2.เอกสารประกอบอื่นๆ เช่น Plan renovate หรือ ใบเสนอราคาที่มีแผนผังชัดเจน



#ช่วงร้านค้าเข้าตกแต่งร้าน หรือก่อสร้างร้าน มีการเก็บเงินประกันหรือไม่ ถ้ามี เก็บอย่างไร	

- ผู้รับเหมาจ่ายเป็นแคชเชียร์เช็คให้กับ constaction team และจะคืนแคชเชียร์เช็คฉบับนั้นให้กับผู้รับเหมาเมื่อสิ้นสุดโครงการ โดยไม่มีการนำแคชเชียร์เช็คไปขึ้นเงินเข้าบัญชี



#กรณีลูกค้าจ่ายเงินประกันเกินกว่า สัญญา จะทำอย่างไรได้บ้าง	

1.สามารถนำเงินเกินไปจ่ายแทนค่าเช่าหรือค่าบริการในสาขานั้นๆ หรือสาขาอื่นภายใต้ชื่อคู่สัญญาเดียวกันได้ โดยLeasing จะต้องเป็นผู้ชี้แจงรายละเอียดและยืนยันกับลูกค้า   

2. Leasing manater ทำเรื่องเปิด PO ขอคืนให้กับผู้เช่าได้<br>

**Node ID:** ecc0aeff-3682-4dc1-87c1-9a5d47deb322<br>**Similarity:** 0.42856094241142273<br>**Text:** สาขา.ผู้ทำสัญญา. ปรากฏว่าได้หายไป”

- ใช้ไม่ได้ เพราะโลตัสออกใบเสร็จให้ลูกค้าทุกเดือน ถ้าไม่ระบุว่าเป็นใบเสร็จเงินประกัน ก็ไม่สามารถใช้ได้

 

#ใบแจ้งความ ระบุเป็น “ใบเสร็จเงินมัดจำ” ได้หรือไม่

- อนุโลมให้ใช้คำว่า “เงินมัดจำ” แทนคำว่า “เงินประกัน” ได้ เพราะสื่อถึงเจตนาเดียวกัน

 

#ใบเสร็จเงินประกันที่แนบมาในชุดเอกสารขอคืนเงินประกัน ใช้เป็นสำเนา ได้หรือไม่

- อนุโลมให้ใช้ได้ สำหรับผู้เช่าที่มีการรับรองสำเนา และระบุสาเหตุที่ไม่สามารถนำส่งฉบับจริงได้ เช่น ต้องใช้ยื่นสรรพากร ต้องเก็บไว้เป็นหลักฐาน หรือต้องใช้สำหรับสัญญาอื่นอีก เป็นต้น

 

#เข้าไปดูเงินประกันในระบบแล้วไม่พบ ถือว่าถูก write off ไหม

- ยังไม่ถือว่าถูก write off ต้องแจ้งข้อมูลมาเพื่อตรวจสอบก่อน คือแจ้ง ชื่อลูกค้า, ร้านค้า, สาขา, unit และสถานะว่ายังเช่าพื้นที่อยู่ หรือสิ้นสุดสัญญาไปแล้ว เพื่อตรวจสอบว่าเงินประกันยังคงมีอยู่หรือไม่อีกครั้ง

 

#การตรวจสอบเงินประกัน ต้องแยกวัตถุประสงค์การตรวจสอบไหม ว่าวัตถุประสงค์ลักษณะนี้ ต้องส่งตรวจสอบกับใคร

- สามารถส่งอีเมลถึง DL_TH-AR-INVOICE@lotuss.<br>